In [ ]:
!apt-get update
!apt-get install -y chromium-chromedriver
!pip install selenium
!pip install lxml
!pip install transformers torch
!pip install groq
!pip install pymongo

In [ ]:
import sys
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options

chrome_driver_path = '/usr/lib/chromium-browser/chromedriver'

chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--remote-debugging-port=9222')

os.environ["PATH"] += f":{chrome_driver_path}"

driver = webdriver.Chrome(options=chrome_options)

In [ ]:
import time
from lxml import html
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from pymongo import MongoClient
import json
from groq import Groq
import os
import re

MONGO_URI = ""
client = MongoClient(MONGO_URI)
db = client[""]
collection = db[""]

os.environ["GROQ_API_KEY"] = ""
groq_api_key = os.getenv("GROQ_API_KEY")
groq_client = Groq(api_key=groq_api_key)

system_prompt = """
Extract the following information from the provided job posting and format it as a JSON object:
{
  "title": "Job Title",
  "description": "Description",
  "min_exp_in_years": "Minimum experience in years (e.g., 0, 1, 2, etc.)",
  "max_exp_in_years": "Maximum experience in years (e.g., 0, 1, 2, etc.)",
  "location": "Location of the job",
  "remote": "True if remote, False otherwise",
  "hybrid": "True if hybrid, False otherwise",
  "on_site": "True if on-site, False otherwise",
  "tags": "List of relevant skills or technologies (e.g., ['python', 'javascript', 'react'])",
  "min_salary": "Minimum salary (if provided)",
  "max_salary": "Maximum salary (if provided)",
  "company_name": "Name of the company",
  "required_skills": "List of required skills (e.g., ['Python', 'JavaScript', 'AWS', 'Azure'])",
  "qualifications": "Minimum qualifications required for the job",
  "where_to_apply": "Instructions on how to apply for the job"
}
"""

def get_job_info_from_llm(extracted_text):
    llm_response = groq_client.chat.completions.create(
        model="llama-3.1-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": extracted_text}
        ],
        temperature=0.3,
        max_tokens=5000
    )
    return llm_response.choices[0].message.content

def extract_json_from_response(response):
    try:
        json_matches = re.findall(r'(\{.*?\}|\[.*?\])', response, re.DOTALL)
        if json_matches:
            job_data = []
            for raw_json in json_matches:
                fixed_json = raw_json.replace("\n", "").replace("\r", "").replace("'", '"').strip()
                print("Fixed JSON String:", fixed_json)
                try:
                    job_data.append(json.loads(fixed_json))
                except json.JSONDecodeError as e:
                    print(f"JSON decoding error: {e}")
                    print(f"Failed to parse JSON: {fixed_json}")
            return job_data if job_data else None
        else:
            print("No JSON object found in response.")
            return None
    except json.JSONDecodeError as e:
        print(f"JSON decoding error: {e}")
        print(f"Failed to parse JSON: {response}")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

def fetch_dynamic_content():
    try:
        driver.get("")
        time.sleep(5)

        rendered_html = driver.page_source
        content = html.fromstring(rendered_html)

        iframes = content.xpath("//iframe[@title='Embedded post']")
        print(f"Found {len(iframes)} iframes with title 'Embedded post'")

        for iframe in iframes:
            iframe_url = iframe.attrib.get('src', '')
            print(f"Fetching content from iframe URL: {iframe_url}")

            driver.get(iframe_url)
            time.sleep(5)

            iframe_rendered_html = driver.page_source
            iframe_content = html.fromstring(iframe_rendered_html)

            p_tag_content = iframe_content.xpath(
                "//div[contains(@class, 'attributed-text-segment-list__container')]//p[contains(@class, 'attributed-text-segment-list__content')]//text()"
            )

            if p_tag_content:
                extracted_text = ' '.join(p_tag_content).strip()
                print(f"Extracted Text: {extracted_text}")

                job_info = get_job_info_from_llm(extracted_text)
                print(f"Response from LLM: {job_info}")

                job_data = extract_json_from_response(job_info)
                if job_data is None:
                    print(f"Failed to parse JSON. Raw Response: {job_info}")

                if job_data:
                    if isinstance(job_data, list):
                        for job in job_data:
                            try:
                                collection.insert_one(job)
                                print("Job information saved to MongoDB:", job)
                            except Exception as e:
                                print(f"Error inserting job to MongoDB: {e}")
                    else:
                        try:
                            collection.insert_one(job_data)
                            print("Job information saved to MongoDB:", job_data)
                        except Exception as e:
                            print(f"Error inserting job to MongoDB: {e}")
                else:
                    print("Failed to extract JSON data.")
            else:
                print("No content found under the p tag.")
    finally:
        driver.quit()

fetch_dynamic_content()